# Visium HD: epithelial programs

The HD fit uses the prepared 32 µm bins in `data/colorectal_cancer/HD_data.npz` and the shared single-cell reference already stored there. `run.py --name HD` writes `Colorectal_cancer/results/HD`. This notebook shows the epithelial mixing weight, every selected epithelial program, and how those activities line up with secretory and proliferation scores computed from genes outside the fitted panel.


In [ ]:
from utils.io import load_npz, load_spatial, repo_root
from utils.metrics import corr, partial_rank_residual
from utils.plots import gene_weights, panel, symmetric_limit, tissue
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = repo_root()
GROUPS = {
    "secretory_goblet_associated": ["MUC2", "TFF3", "SPINK4", "CLCA1", "ZG16", "AGR2", "REG4"],
    "proliferation": ["MKI67", "TOP2A", "PCNA", "MCM2", "MCM5", "TYMS"],
}
spatial = load_spatial(ROOT / "data/colorectal_cancer/HD_data.npz")
fit = load_npz(ROOT / "Colorectal_cancer/results/HD/final.npz")
markers = load_npz(ROOT / "data/colorectal_cancer/hd32_markers.npz")
epithelial = list(fit["cell_types"]).index("Epithelial")
programs = np.flatnonzero(fit["program_cell_type"] == epithelial)
pi = fit["pi"][:, epithelial]
keep = pi >= 0.05
logged = np.log1p(10000 * markers["counts"] / markers["library_size_all_genes"][:, None])
marker_genes = list(markers["genes"])
scores = {name: np.mean([logged[:, marker_genes.index(gene)] for gene in genes], axis=0) for name, genes in GROUPS.items()}
library = markers["library_size_all_genes"]


In [ ]:
coords = spatial["coords"]
n = len(programs)
fig = plt.figure(figsize=(2.3 * (n + 1), 7.4))
gs = fig.add_gridspec(3, n + 1, hspace=.45, wspace=.25)
letters = iter("abcdefghijklmnopqrstuvwxyz")
ax = fig.add_subplot(gs[0, 0])
image = tissue(ax, coords, pi, "Epithelial weight", lim=(0, 1), cmap="viridis", size=.35)
panel(ax, next(letters))
fig.colorbar(image, ax=ax, orientation="horizontal", fraction=.04, pad=.04)
for j, program in enumerate(programs):
    ax = fig.add_subplot(gs[0, j + 1])
    activity = fit["h_mean"][program]
    limit = symmetric_limit(activity, keep)
    image = tissue(ax, coords, activity, f"P{int(fit['program_index'][program]) + 1}", mask=keep, lim=(-limit, limit), size=.35)
    panel(ax, next(letters))
    fig.colorbar(image, ax=ax, orientation="horizontal", fraction=.04, pad=.04)
for j, (name, title) in enumerate([("secretory_goblet_associated", "Secretory score"), ("proliferation", "Proliferation score")]):
    ax = fig.add_subplot(gs[1, j])
    image = tissue(ax, coords, scores[name], title, cmap="viridis", size=.35)
    panel(ax, next(letters))
    fig.colorbar(image, ax=ax, orientation="horizontal", fraction=.04, pad=.04)
rows = []
for program in programs:
    activity = fit["h_mean"][program]
    for name, score in scores.items():
        rows.append(dict(
            program=int(fit["program_index"][program]) + 1, target=name,
            partial_rank_correlation=corr(
                partial_rank_residual(activity[keep], pi[keep], library[keep]),
                partial_rank_residual(score[keep], pi[keep], library[keep]))))
frame = pd.DataFrame(rows)
ax = fig.add_subplot(gs[2, :])
for j, (name, title) in enumerate([("secretory_goblet_associated", "Secretory"), ("proliferation", "Proliferation")]):
    part = frame[frame.target == name].sort_values("program")
    ax.barh(np.arange(len(part)) + (j - .5) * .32, part.partial_rank_correlation, height=.3, color=["#82AE9E", "#CC956B"][j], label=title)
ax.set_yticks(range(n), [f"P{int(fit['program_index'][p]) + 1}" for p in programs])
ax.axvline(0, color="#888888", lw=.6)
ax.set_xlim(-1, 1)
ax.set_xlabel("Partial rank correlation")
ax.set_title(f"Epithelial weight ≥ 0.05, n = {int(keep.sum()):,}")
ax.legend(fontsize=7, loc="lower right")
panel(ax, next(letters))
fig.tight_layout()
fig


In [ ]:
weight_figure, weight_axes = plt.subplots(1, n, figsize=(3.3 * n, 3.8), squeeze=False)
for ax, program in zip(weight_axes.flat, programs):
    gene_weights(ax, fit["loadings"][program], fit["genes"], f"P{int(fit['program_index'][program]) + 1}", n=15)
weight_figure.tight_layout()
frame
